# 01 — Nigerian Education Dataset Exploration

**Research:** Adaptive LLM for Curriculum-Aligned Educational Content in African Contexts

This notebook explores all 7 Nigerian education HuggingFace datasets used as the RAG corpus.
For each dataset we examine:
- Schema and column names
- Row counts and text length distributions
- Subject / topic / level distributions
- Sample rows
- Vocabulary overlap with curriculum board keywords

In [ ]:
!pip install -q datasets pandas matplotlib seaborn wordcloud

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 120)

In [ ]:
# Dataset registry — mirrors research/datasets/dataset_registry.yaml
DATASETS = {
    'vocational':            'electricsheepafrica/nigerian_education_vocational_technical',
    'continuous_assessment': 'Ben-45/nigerian_education_continuous_assessment',
    'digital_learning':      'electricsheepafrica/nigerian_education_digital_learning',
    'online_exams':          'electricsheepafrica/nigerian_education_online_exams',
    'learning_materials':    'electricsheepafrica/nigerian_education_learning_materials',
    'teacher_training':      'electricsheepafrica/nigerian_education_teacher_training',
    'special_needs':         'electricsheepafrica/nigerian_education_special_needs',
}

MAX_ROWS = 2000  # cap per dataset for exploration

## Load all datasets

In [ ]:
dfs = {}
for key, hf_id in DATASETS.items():
    try:
        ds = load_dataset(hf_id, split='train', streaming=True)
        rows = []
        for i, row in enumerate(ds):
            if i >= MAX_ROWS: break
            rows.append(row)
        dfs[key] = pd.DataFrame(rows)
        print(f'✓ {key:<25} {len(dfs[key]):>6} rows | columns: {list(dfs[key].columns)}')
    except Exception as e:
        print(f'✗ {key:<25} ERROR: {e}')
        dfs[key] = pd.DataFrame()

## Schema summary

In [ ]:
summary = []
for key, df in dfs.items():
    if df.empty: continue
    # Find the best text column
    text_cols = [c for c in df.columns if df[c].dtype == object]
    best_col = max(text_cols, key=lambda c: df[c].str.len().median() if df[c].dtype == object else 0, default=None)
    median_len = df[best_col].str.len().median() if best_col else 0
    summary.append({
        'dataset': key,
        'rows': len(df),
        'columns': len(df.columns),
        'column_names': ', '.join(df.columns.tolist()[:8]),
        'best_text_col': best_col,
        'median_text_len': round(median_len),
    })

pd.DataFrame(summary).set_index('dataset')

## Sample rows per dataset

In [ ]:
for key, df in dfs.items():
    if df.empty: continue
    print(f'\n{"="*60}')
    print(f'  {key}')
    print(f'{"="*60}')
    display(df.head(3))

## Text length distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, (key, df) in enumerate(dfs.items()):
    ax = axes[i]
    if df.empty:
        ax.set_title(f'{key}\n(load failed)')
        continue
    # Use all text-like columns concatenated
    text_lengths = df.apply(lambda row: len(' '.join(str(v) for v in row.values if isinstance(v, str))), axis=1)
    ax.hist(text_lengths, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{key}\n(n={len(df)}, median={text_lengths.median():.0f} chars)', fontsize=9)
    ax.set_xlabel('Text length (chars)')
    ax.set_ylabel('Count')

# Hide extra subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Text Length Distributions — Nigerian Education Datasets', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../evaluation/results/text_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Subject distribution (where available)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, (key, df) in enumerate(dfs.items()):
    ax = axes[i]
    if df.empty or 'subject' not in df.columns:
        ax.set_title(f'{key}\n(no subject col)')
        continue
    counts = df['subject'].value_counts().head(10)
    counts.plot(kind='barh', ax=ax, color='coral', edgecolor='white')
    ax.set_title(f'{key}', fontsize=9)
    ax.set_xlabel('Count')
    ax.invert_yaxis()

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Subject Distributions — Nigerian Education Datasets', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../evaluation/results/subject_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Curriculum keyword coverage

Measures how many NERDC/WAEC alignment keywords are present across the corpus.
Higher coverage → better RAG retrieval quality for curriculum-aligned generation.

In [ ]:
NERDC_KEYWORDS = ['objective', 'activity', 'evaluation', 'scheme', 'term', 'week', 'curriculum']
WAEC_KEYWORDS  = ['question', 'mark', 'answer', 'examination', 'candidate', 'score']

keyword_coverage = []
for key, df in dfs.items():
    if df.empty: continue
    combined = df.apply(
        lambda row: ' '.join(str(v).lower() for v in row.values if isinstance(v, str)), axis=1
    ).str.cat(sep=' ')
    nerdc_hits = sum(1 for kw in NERDC_KEYWORDS if kw in combined)
    waec_hits  = sum(1 for kw in WAEC_KEYWORDS  if kw in combined)
    keyword_coverage.append({
        'dataset': key,
        'nerdc_keyword_coverage': f'{nerdc_hits}/{len(NERDC_KEYWORDS)}',
        'waec_keyword_coverage':  f'{waec_hits}/{len(WAEC_KEYWORDS)}',
        'nerdc_score': nerdc_hits / len(NERDC_KEYWORDS),
        'waec_score':  waec_hits / len(WAEC_KEYWORDS),
    })

kdf = pd.DataFrame(keyword_coverage).set_index('dataset')
display(kdf)

# Plot
kdf[['nerdc_score', 'waec_score']].plot(kind='bar', figsize=(10, 4), color=['steelblue', 'coral'], edgecolor='white')
plt.title('Curriculum Keyword Coverage by Dataset')
plt.ylabel('Coverage ratio')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../evaluation/results/keyword_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## Save dataset statistics

In [ ]:
import json
from pathlib import Path

stats = {}
for key, df in dfs.items():
    if df.empty:
        stats[key] = {'status': 'failed'}
        continue
    text_lengths = df.apply(
        lambda row: len(' '.join(str(v) for v in row.values if isinstance(v, str))), axis=1
    )
    stats[key] = {
        'rows_loaded': len(df),
        'columns': df.columns.tolist(),
        'text_len_mean': round(text_lengths.mean(), 1),
        'text_len_median': round(text_lengths.median(), 1),
        'text_len_min': int(text_lengths.min()),
        'text_len_max': int(text_lengths.max()),
        'has_subject_col': 'subject' in df.columns,
    }

out_path = Path('../evaluation/results/dataset_stats.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(stats, indent=2))
print(f'Saved to {out_path}')
print(json.dumps(stats, indent=2))